In [ ]:
# 📦 Imports
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

# 📁 Paths
DATA_PATH = "../data/netflix_customer_churn.csv"
MODEL_DIR = "../backend/models/netflix/"
MODEL_NAME = "ctr_model_xgb.pkl"
FEATURES_NAME = "feature_names_ctr.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# 📊 Load dataset
df = pd.read_csv(DATA_PATH)
print("✅ Loaded Netflix dataset with shape:", df.shape)

# 🎯 Define CTR Label (Simulated: top 25% watch_time → clicked = 1)
threshold = df["avg_watch_time_per_day"].quantile(0.75)
df["clicked"] = (df["avg_watch_time_per_day"] >= threshold).astype(int)

# 🧹 Preprocessing
LABEL_COL = "clicked"
y = df[LABEL_COL]
X = df.drop(columns=["customer_id", "churned", LABEL_COL])  # Remove identifiers and unrelated labels

# One-hot encode categoricals
X = pd.get_dummies(X)

# 💾 Save feature names
feature_names = list(X.columns)
with open(os.path.join(MODEL_DIR, FEATURES_NAME), "w") as f:
    json.dump(feature_names, f, indent=2)

# 📉 Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 🧠 Train XGBoost Model
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=feature_names)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}
num_rounds = 100

bst = xgb.train(params, dtrain, num_boost_round=num_rounds)

# 🔍 Evaluate
y_pred_probs = bst.predict(dtest)
y_pred = (y_pred_probs > 0.5).astype(int)

report = classification_report(y_test, y_pred, digits=4)
conf_matrix = confusion_matrix(y_test, y_pred)

print("📋 Classification Report:\n", report)
print("🧩 Confusion Matrix:\n", conf_matrix)

# 💾 Save model
model_path = os.path.join(MODEL_DIR, MODEL_NAME)
bst.save_model(model_path)
print(f"✅ Model saved to: {model_path}")
print(f"🧠 Feature names saved to: {os.path.join(MODEL_DIR, FEATURES_NAME)}")
print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S")) 

✅ Loaded Netflix dataset with shape: (5000, 14)
📋 Classification Report:
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       747
           1     1.0000    1.0000    1.0000       253

    accuracy                         1.0000      1000
   macro avg     1.0000    1.0000    1.0000      1000
weighted avg     1.0000    1.0000    1.0000      1000

🧩 Confusion Matrix:
 [[747   0]
 [  0 253]]
✅ Model saved to: ../backend/models/netflix/ctr_model_xgb.pkl
🧠 Feature names saved to: ../backend/models/netflix/feature_names_ctr.json
🏁 Done at 2026-01-24 12:37:43


/var/folders/91/syqbgxdn69n01td9pvbx3dz80000gn/T/ipykernel_34761/1866060780.py:74: UserWarning: [12:37:43] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  bst.save_model(model_path)
